1 ВАРИАНТ КОДА

In [ ]:
"""
Модуль для распознавания марки автомобиля по фотографии.
"""

import os
import logging
from typing import Dict, List, Optional, Tuple
import numpy as np
from PIL import Image
import requests

# Импорт библиотек для глубокого обучения
try:
    import torch
    import torchvision.transforms as transforms
    from torchvision import models
    import torch.nn as nn
except ImportError:
    print("Установите библиотеки для глубокого обучения:")
    print("pip install torch torchvision pillow requests")


# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class CarBrandClassifier:
    """Классификатор марок автомобилей."""

    # Пример списка марок автомобилей (можно расширить)
    CAR_BRANDS = [
        'audi', 'bmw', 'chevrolet', 'ford', 'honda',
        'hyundai', 'mercedes', 'nissan', 'toyota', 'volkswagen'
    ]

    def __init__(self, model_path: Optional[str] = None):
        """
        Инициализация классификатора.

        Args:
            model_path: Путь к предобученной модели (опционально)
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Используется устройство: {self.device}")

        self.model = self._load_model(model_path)
        self.transform = self._get_transforms()

    def _load_model(self, model_path: Optional[str]) -> nn.Module:
        """
        Загрузка модели.

        Args:
            model_path: Путь к модели

        Returns:
            Загруженная модель PyTorch
        """
        # Используем предобученную ResNet
        model = models.resnet50(pretrained=True)

        # Заменяем последний слой для нашего числа классов
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, len(self.CAR_BRANDS))

        if model_path and os.path.exists(model_path):
            try:
                model.load_state_dict(torch.load(model_path, map_location=self.device))
                logger.info(f"Модель загружена из {model_path}")
            except Exception as e:
                logger.warning(f"Не удалось загрузить модель: {e}")
                logger.info("Используется модель с случайными весами")
        else:
            logger.info("Используется модель с предобученными весами (требуется дообучение)")

        model = model.to(self.device)
        model.eval()

        return model

    def _get_transforms(self) -> transforms.Compose:
        """
        Получение трансформаций для изображений.

        Returns:
            Композиция трансформаций
        """
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def preprocess_image(self, image_path: str) -> torch.Tensor:
        """
        Предобработка изображения.

        Args:
            image_path: Путь к изображению

        Returns:
            Тензор с обработанным изображением
        """
        try:
            image = Image.open(image_path).convert('RGB')
            image = self.transform(image)
            image = image.unsqueeze(0)  # Добавляем размер батча
            return image.to(self.device)
        except Exception as e:
            logger.error(f"Ошибка при загрузке изображения: {e}")
            raise

    def predict(self, image_path: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Предсказание марки автомобиля по изображению.

        Args:
            image_path: Путь к изображению
            top_k: Количество лучших предсказаний

        Returns:
            Список кортежей (марка, вероятность)
        """
        try:
            # Предобработка изображения
            image_tensor = self.preprocess_image(image_path)

            # Предсказание
            with torch.no_grad():
                outputs = self.model(image_tensor)
                probabilities = torch.nn.functional.softmax(outputs, dim=1)
                top_probs, top_indices = torch.topk(probabilities, top_k)

            # Преобразование в читаемый формат
            results = []
            for i in range(top_k):
                brand = self.CAR_BRANDS[top_indices[0][i].item()]
                prob = top_probs[0][i].item() * 100
                results.append((brand, prob))

            return results

        except Exception as e:
            logger.error(f"Ошибка при предсказании: {e}")
            return []

    def predict_from_url(self, image_url: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Предсказание марки автомобиля по URL изображения.

        Args:
            image_url: URL изображения
            top_k: Количество лучших предсказаний

        Returns:
            Список кортежей (марка, вероятность)
        """
        try:
            # Загрузка изображения из URL
            response = requests.get(image_url, stream=True, timeout=10)
            response.raise_for_status()

            # Сохранение во временный файл
            temp_path = "temp_image.jpg"
            with open(temp_path, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)

            # Предсказание
            results = self.predict(temp_path, top_k)

            # Удаление временного файла
            os.remove(temp_path)

            return results

        except Exception as e:
            logger.error(f"Ошибка при загрузке изображения по URL: {e}")
            return []


def train_model(
    data_dir: str,
    epochs: int = 10,
    batch_size: int = 32,
    model_save_path: str = "car_brand_classifier.pth"
) -> None:
    """
    Функция для обучения модели на своем датасете.

    Args:
        data_dir: Директория с данными (должна содержать поддиректории с марками)
        epochs: Количество эпох обучения
        batch_size: Размер батча
        model_save_path: Путь для сохранения модели
    """
    try:
        from torch.utils.data import DataLoader, Dataset
        from torchvision.datasets import ImageFolder

        logger.info("Начало обучения модели...")

        # Определение трансформаций
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # Загрузка данных
        train_dataset = ImageFolder(
            os.path.join(data_dir, 'train'),
            transform=train_transform
        )
        val_dataset = ImageFolder(
            os.path.join(data_dir, 'val'),
            transform=val_transform
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False
        )

        # Инициализация модели
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = models.resnet50(pretrained=True)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, len(train_dataset.classes))
        model = model.to(device)

        # Функция потерь и оптимизатор
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

        # Обучение
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            # Валидация
            model.eval()
            correct = 0
            total = 0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            accuracy = 100 * correct / total
            logger.info(
                f"Эпоха {epoch+1}/{epochs}, "
                f"Потери: {running_loss/len(train_loader):.4f}, "
                f"Точность: {accuracy:.2f}%"
            )

        # Сохранение модели
        torch.save(model.state_dict(), model_save_path)
        logger.info(f"Модель сохранена в {model_save_path}")

    except ImportError:
        logger.error("Для обучения требуется установить torch и torchvision")
    except Exception as e:
        logger.error(f"Ошибка при обучении: {e}")


def main() -> None:
    """Основная функция для демонстрации работы."""

    print("=" * 50)
    print("Классификатор марок автомобилей")
    print("=" * 50)

    # Инициализация классификатора
    classifier = CarBrandClassifier()

    # Пример использования
    while True:
        print("\nВыберите опцию:")
        print("1. Распознать автомобиль по локальному изображению")
        print("2. Распознать автомобиль по URL изображения")
        print("3. Выйти")

        choice = input("Введите номер опции: ").strip()

        if choice == '1':
            image_path = input("Введите путь к изображению: ").strip()
            if os.path.exists(image_path):
                results = classifier.predict(image_path)
                if results:
                    print("\nРезультаты распознавания:")
                    for brand, prob in results:
                        print(f"  {brand}: {prob:.2f}%")
                else:
                    print("Не удалось распознать изображение")
            else:
                print("Файл не найден")

        elif choice == '2':
            image_url = input("Введите URL изображения: ").strip()
            results = classifier.predict_from_url(image_url)
            if results:
                print("\nРезультаты распознавания:")
                for brand, prob in results:
                    print(f"  {brand}: {prob:.2f}%")
            else:
                print("Не удалось распознать изображение")

        elif choice == '3':
            print("Выход...")
            break

        else:
            print("Неверный выбор, попробуйте снова")


if __name__ == "__main__":
    # Для обучения модели на своем датасете:
    # train_model("path/to/your/dataset", epochs=10)

    main()

Классификатор марок автомобилей

Выберите опцию:
1. Распознать автомобиль по локальному изображению
2. Распознать автомобиль по URL изображения
3. Выйти


2 ВАРИАНТ КОДА

In [ ]:
"""
Модуль для распознавания марки автомобиля по фотографии с высокой точностью.
"""

import os
import logging
import json
from typing import Dict, List, Optional, Tuple, Any
import numpy as np
from PIL import Image, ImageOps, ImageEnhance
import requests
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Импорт библиотек для глубокого обучения
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
    import torchvision.transforms as transforms
    from torchvision import models
    from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
    from torch.cuda.amp import GradScaler, autocast
    import timm  # Библиотека с современными моделями
    from sklearn.metrics import classification_report, confusion_matrix
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
except ImportError as e:
    print("Установите необходимые библиотеки:")
    print("pip install torch torchvision timm scikit-learn pandas seaborn matplotlib pillow requests")
    print(f"Ошибка импорта: {e}")


# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('car_classifier.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class ModelType(Enum):
    """Типы доступных моделей."""
    EFFICIENTNET_B4 = 'efficientnet_b4'
    RESNEXT101 = 'resnext101_64x4d'
    VIT_BASE = 'vit_base_patch16_224'
    SWIN_BASE = 'swin_base_patch4_window7_224'
    ENSEMBLE = 'ensemble'


@dataclass
class ModelConfig:
    """Конфигурация модели."""
    model_type: ModelType = ModelType.EFFICIENTNET_B4
    img_size: int = 384
    batch_size: int = 32
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    dropout_rate: float = 0.3
    use_amp: bool = True  # Automatic Mixed Precision


class AdvancedAugmentations:
    """Расширенные аугментации для обучения."""

    @staticmethod
    def get_train_transforms(img_size: int = 384):
        """Трансформации для обучения."""
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            ),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1)
            ),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    @staticmethod
    def get_val_transforms(img_size: int = 384):
        """Трансформации для валидации/тестирования."""
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


class CarBrandClassifier:
    """Классификатор марок автомобилей с высокой точностью."""

    # Расширенный список марок автомобилей
    CAR_BRANDS = [
        'audi', 'bmw', 'chevrolet', 'ford', 'honda',
        'hyundai', 'mercedes', 'nissan', 'toyota', 'volkswagen',
        'kia', 'mazda', 'lexus', 'subaru', 'jeep',
        'tesla', 'porsche', 'ferrari', 'lamborghini', 'maserati'
    ]

    # Мэппинг для альтернативных названий
    BRAND_ALIASES = {
        'mercedes-benz': 'mercedes',
        'mercedes benz': 'mercedes',
        'vw': 'volkswagen',
        'chevy': 'chevrolet',
        'toyota lexus': 'lexus',
        'acura': 'honda',
        'infiniti': 'nissan'
    }

    def __init__(self,
                 model_config: Optional[ModelConfig] = None,
                 model_paths: Optional[List[str]] = None,
                 use_ensemble: bool = True):
        """
        Инициализация классификатора.

        Args:
            model_config: Конфигурация модели
            model_paths: Список путей к предобученным моделям
            use_ensemble: Использовать ансамбль моделей
        """
        self.device = self._get_device()
        logger.info(f"Используется устройство: {self.device}")

        self.model_config = model_config or ModelConfig()
        self.use_ensemble = use_ensemble

        if use_ensemble and model_paths:
            self.models = self._load_ensemble(model_paths)
        else:
            self.model = self._load_model(model_paths[0] if model_paths else None)

        self.transform = AdvancedAugmentations.get_val_transforms(
            self.model_config.img_size
        )
        self.scaler = GradScaler() if self.model_config.use_amp else None

    def _get_device(self):
        """Получение доступного устройства."""
        if torch.cuda.is_available():
            return torch.device('cuda')
        elif torch.backends.mps.is_available():
            return torch.device('mps')
        else:
            return torch.device('cpu')

    def _create_model(self, model_type: ModelType, num_classes: int) -> nn.Module:
        """Создание модели заданного типа."""
        if model_type == ModelType.EFFICIENTNET_B4:
            model = timm.create_model(
                'tf_efficientnet_b4_ns',
                pretrained=True,
                num_classes=num_classes,
                drop_rate=self.model_config.dropout_rate
            )
        elif model_type == ModelType.RESNEXT101:
            model = timm.create_model(
                'resnext101_64x4d',
                pretrained=True,
                num_classes=num_classes
            )
        elif model_type == ModelType.VIT_BASE:
            model = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True,
                num_classes=num_classes
            )
        elif model_type == ModelType.SWIN_BASE:
            model = timm.create_model(
                'swin_base_patch4_window7_224',
                pretrained=True,
                num_classes=num_classes
            )
        else:
            raise ValueError(f"Неизвестный тип модели: {model_type}")

        return model.to(self.device)

    def _load_model(self, model_path: Optional[str]) -> nn.Module:
        """Загрузка одиночной модели."""
        model = self._create_model(self.model_config.model_type, len(self.CAR_BRANDS))

        if model_path and os.path.exists(model_path):
            try:
                state_dict = torch.load(model_path, map_location=self.device)

                # Обработка разных форматов сохранения модели
                if 'model_state_dict' in state_dict:
                    model.load_state_dict(state_dict['model_state_dict'])
                elif 'state_dict' in state_dict:
                    model.load_state_dict(state_dict['state_dict'])
                else:
                    model.load_state_dict(state_dict)

                logger.info(f"Модель загружена из {model_path}")

                if 'accuracy' in state_dict:
                    logger.info(f"Точность модели: {state_dict['accuracy']:.2f}%")

            except Exception as e:
                logger.warning(f"Не удалось загрузить модель: {e}")
                logger.info("Используется предобученная модель")
        else:
            logger.info("Используется предобученная модель ImageNet")

        model.eval()
        return model

    def _load_ensemble(self, model_paths: List[str]) -> List[nn.Module]:
        """Загрузка ансамбля моделей."""
        models = []

        for i, path in enumerate(model_paths):
            if os.path.exists(path):
                try:
                    # Создаем разные архитектуры для разнообразия
                    model_types = [
                        ModelType.EFFICIENTNET_B4,
                        ModelType.RESNEXT101,
                        ModelType.VIT_BASE
                    ]

                    model_type = model_types[i % len(model_types)]
                    model = self._create_model(model_type, len(self.CAR_BRANDS))

                    state_dict = torch.load(path, map_location=self.device)

                    # Загрузка весов с обработкой разных форматов
                    model_state_dict = None
                    for key in ['model_state_dict', 'state_dict']:
                        if key in state_dict:
                            model_state_dict = state_dict[key]
                            break

                    if model_state_dict is None:
                        model_state_dict = state_dict

                    # Изменение размеров последнего слоя при необходимости
                    if 'fc.weight' in model_state_dict:
                        fc_weight = model_state_dict['fc.weight']
                        if fc_weight.shape[0] != len(self.CAR_BRANDS):
                            model.fc = nn.Linear(model.fc.in_features, len(self.CAR_BRANDS))

                    model.load_state_dict(model_state_dict, strict=False)
                    model.eval()
                    models.append(model)

                    logger.info(f"Модель {i+1} загружена из {path}")

                except Exception as e:
                    logger.warning(f"Не удалось загрузить модель {path}: {e}")

        if not models:
            raise ValueError("Не удалось загрузить ни одну модель для ансамбля")

        logger.info(f"Ансамбль из {len(models)} моделей успешно загружен")
        return models

    def preprocess_image(self, image_path: str) -> torch.Tensor:
        """
        Расширенная предобработка изображения.

        Args:
            image_path: Путь к изображению

        Returns:
            Тензор с обработанным изображением
        """
        try:
            # Открытие изображения
            image = Image.open(image_path).convert('RGB')

            # Автоматическая коррекция
            image = ImageOps.exif_transpose(image)  # Исправление ориентации

            # Улучшение контраста
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(1.2)

            # Применение трансформаций
            image = self.transform(image)
            image = image.unsqueeze(0)  # Добавляем размер батча

            return image.to(self.device)

        except Exception as e:
            logger.error(f"Ошибка при загрузке изображения {image_path}: {e}")
            raise

    def _ensemble_predict(self, image_tensor: torch.Tensor, top_k: int = 3) -> List[Tuple[str, float]]:
        """Предсказание с использованием ансамбля моделей."""
        all_probs = []

        for model in self.models:
            with torch.no_grad():
                if self.model_config.use_amp:
                    with autocast():
                        outputs = model(image_tensor)
                else:
                    outputs = model(image_tensor)

                probs = torch.nn.functional.softmax(outputs, dim=1)
                all_probs.append(probs)

        # Усреднение вероятностей
        avg_probs = torch.mean(torch.stack(all_probs), dim=0)
        top_probs, top_indices = torch.topk(avg_probs, min(top_k, len(self.CAR_BRANDS)))

        results = []
        for i in range(top_probs.shape[1]):
            brand = self.CAR_BRANDS[top_indices[0][i].item()]
            prob = top_probs[0][i].item() * 100
            results.append((brand, prob))

        return results

    def predict(self,
                image_path: str,
                top_k: int = 3,
                confidence_threshold: float = 0.0) -> List[Tuple[str, float]]:
        """
        Предсказание марки автомобиля по изображению.

        Args:
            image_path: Путь к изображению
            top_k: Количество лучших предсказаний
            confidence_threshold: Порог уверенности

        Returns:
            Список кортежей (марка, вероятность)
        """
        try:
            # Предобработка изображения
            image_tensor = self.preprocess_image(image_path)

            # Предсказание
            if self.use_ensemble and hasattr(self, 'models'):
                results = self._ensemble_predict(image_tensor, top_k)
            else:
                with torch.no_grad():
                    if self.model_config.use_amp:
                        with autocast():
                            outputs = self.model(image_tensor)
                    else:
                        outputs = self.model(image_tensor)

                    probabilities = torch.nn.functional.softmax(outputs, dim=1)
                    top_probs, top_indices = torch.topk(
                        probabilities,
                        min(top_k, len(self.CAR_BRANDS))
                    )

                # Преобразование в читаемый формат
                results = []
                for i in range(top_probs.shape[1]):
                    brand = self.CAR_BRANDS[top_indices[0][i].item()]
                    prob = top_probs[0][i].item() * 100
                    if prob >= confidence_threshold:
                        results.append((brand, prob))

            return results

        except Exception as e:
            logger.error(f"Ошибка при предсказании для {image_path}: {e}")
            return []

    def predict_batch(self,
                     image_paths: List[str],
                     batch_size: int = 16) -> Dict[str, List[Tuple[str, float]]]:
        """
        Пакетное предсказание для нескольких изображений.

        Args:
            image_paths: Список путей к изображениям
            batch_size: Размер батча

        Returns:
            Словарь с результатами для каждого изображения
        """
        results = {}

        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_tensors = []
            valid_paths = []

            for path in batch_paths:
                try:
                    tensor = self.preprocess_image(path)
                    batch_tensors.append(tensor)
                    valid_paths.append(path)
                except Exception as e:
                    logger.warning(f"Пропущено изображение {path}: {e}")

            if batch_tensors:
                batch = torch.cat(batch_tensors, dim=0)

                with torch.no_grad():
                    if self.model_config.use_amp:
                        with autocast():
                            outputs = self.model(batch) if not self.use_ensemble else None
                    else:
                        outputs = self.model(batch) if not self.use_ensemble else None

                    if outputs is not None:
                        probabilities = torch.nn.functional.softmax(outputs, dim=1)
                        top_probs, top_indices = torch.topk(probabilities, 1)

                        for j, path in enumerate(valid_paths):
                            brand = self.CAR_BRANDS[top_indices[j].item()]
                            prob = top_probs[j].item() * 100
                            results[path] = [(brand, prob)]

        return results

    def predict_from_url(self,
                        image_url: str,
                        top_k: int = 3,
                        timeout: int = 30) -> List[Tuple[str, float]]:
        """
        Предсказание марки автомобиля по URL изображения.

        Args:
            image_url: URL изображения
            top_k: Количество лучших предсказаний
            timeout: Таймаут загрузки

        Returns:
            Список кортежей (марка, вероятность)
        """
        try:
            # Загрузка изображения из URL
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }

            response = requests.get(
                image_url,
                headers=headers,
                stream=True,
                timeout=timeout
            )
            response.raise_for_status()

            # Сохранение во временный файл
            temp_path = f"temp_image_{hash(image_url)}.jpg"
            with open(temp_path, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)

            # Предсказание
            results = self.predict(temp_path, top_k)

            # Удаление временного файла
            os.remove(temp_path)

            return results

        except Exception as e:
            logger.error(f"Ошибка при загрузке изображения по URL {image_url}: {e}")
            return []

    def evaluate_model(self, test_dir: str) -> Dict[str, Any]:
        """
        Оценка модели на тестовом наборе данных.

        Args:
            test_dir: Директория с тестовыми данными

        Returns:
            Словарь с метриками оценки
        """
        from sklearn.metrics import accuracy_score, precision_recall_fscore_support

        all_preds = []
        all_labels = []
        all_probs = []

        # Загрузка тестовых данных
        transform = AdvancedAugmentations.get_val_transforms(self.model_config.img_size)
        test_dataset = ImageFolder(test_dir, transform=transform)
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.model_config.batch_size,
            shuffle=False
        )

        self.model.eval()

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                if self.model_config.use_amp:
                    with autocast():
                        outputs = self.model(images)
                else:
                    outputs = self.model(images)

                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        # Вычисление метрик
        accuracy = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels,
            all_preds,
            average='weighted'
        )

        # Матрица ошибок
        cm = confusion_matrix(all_labels, all_preds)

        # Отчет по классификации
        class_report = classification_report(
            all_labels,
            all_preds,
            target_names=test_dataset.classes,
            output_dict=True
        )

        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': cm.tolist(),
            'classification_report': class_report,
            'test_size': len(test_dataset)
        }

        logger.info(f"Точность на тестовом наборе: {accuracy:.4f}")
        logger.info(f"F1-мера: {f1:.4f}")

        return metrics

    def save_predictions(self,
                        predictions: Dict[str, List[Tuple[str, float]]],
                        output_path: str = 'predictions.csv'):
        """
        Сохранение предсказаний в файл.

        Args:
            predictions: Словарь с предсказаниями
            output_path: Путь для сохранения
        """
        import pandas as pd

        data = []
        for image_path, preds in predictions.items():
            for brand, prob in preds:
                data.append({
                    'image_path': image_path,
                    'predicted_brand': brand,
                    'confidence': prob
                })

        df = pd.DataFrame(data)
        df.to_csv(output_path, index=False)
        logger.info(f"Предсказания сохранены в {output_path}")

    def visualize_predictions(self,
                             image_path: str,
                             save_path: Optional[str] = None):
        """
        Визуализация предсказаний для изображения.

        Args:
            image_path: Путь к изображению
            save_path: Путь для сохранения визуализации
        """
        try:
            import matplotlib.pyplot as plt
            from matplotlib import patches

            # Получение предсказаний
            predictions = self.predict(image_path, top_k=5)

            # Загрузка и отображение изображения
            image = Image.open(image_path).convert('RGB')
            image = ImageOps.exif_transpose(image)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

            # Отображение изображения
            ax1.imshow(image)
            ax1.axis('off')
            ax1.set_title('Исходное изображение')

            # Отображение предсказаний
            brands = [p[0] for p in predictions]
            probs = [p[1] for p in predictions]

            colors = plt.cm.RdYlGn(probs / 100)
            bars = ax2.barh(brands, probs, color=colors)
            ax2.set_xlabel('Вероятность (%)')
            ax2.set_title('Предсказания модели')
            ax2.set_xlim([0, 100])

            # Добавление значений на столбцы
            for bar, prob in zip(bars, probs):
                width = bar.get_width()
                ax2.text(width + 1, bar.get_y() + bar.get_height()/2,
                        f'{prob:.1f}%', va='center')

            plt.tight_layout()

            if save_path:
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                logger.info(f"Визуализация сохранена в {save_path}")

            plt.show()

        except Exception as e:
            logger.error(f"Ошибка при визуализации: {e}")


class AdvancedTrainer:
    """Продвинутый тренер для обучения моделей."""

    def __init__(self, config: ModelConfig):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.scaler = GradScaler() if config.use_amp else None

    def train(self,
              train_dir: str,
              val_dir: str,
              num_epochs: int = 50,
              patience: int = 10,
              model_save_path: str = 'best_model.pth'):
        """
        Обучение модели с продвинутыми техниками.

        Args:
            train_dir: Директория с обучающими данными
            val_dir: Директория с валидационными данными
            num_epochs: Количество эпох
            patience: Количество эпох для ранней остановки
            model_save_path: Путь для сохранения лучшей модели
        """
        # Загрузка данных
        train_transform = AdvancedAugmentations.get_train_transforms(self.config.img_size)
        val_transform = AdvancedAugmentations.get_val_transforms(self.config.img_size)

        train_dataset = ImageFolder(train_dir, transform=train_transform)
        val_dataset = ImageFolder(val_dir, transform=val_transform)

        # Взвешенный семплер для несбалансированных данных
        class_counts = np.bincount([label for _, label in train_dataset.samples])
        class_weights = 1. / class_counts
        sample_weights = class_weights[train_dataset.targets]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            sampler=sampler,
            num_workers=4,
            pin_memory=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        # Создание модели
        model = timm.create_model(
            'tf_efficientnet_b4_ns',
            pretrained=True,
            num_classes=len(train_dataset.classes),
            drop_rate=self.config.dropout_rate
        ).to(self.device)

        # Функция потерь с весами классов
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

        # Оптимизатор и шедулер
        optimizer = optim.AdamW(
            model.parameters(),
            lr=self.config.learning_rate,
            weight_decay=self.config.weight_decay
        )

        scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

        # Обучение
        best_val_acc = 0.0
        patience_counter = 0

        for epoch in range(num_epochs):
            # Фаза обучения
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0

            for images, labels in train_loader:
                images, labels = images.to(self.device), labels.to(self.device)

                optimizer.zero_grad()

                if self.config.use_amp:
                    with autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                    self.scaler.scale(loss).backward()
                    self.scaler.step(optimizer)
                    self.scaler.update()
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

                train_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()

            train_acc = 100 * train_correct / train_total

            # Фаза валидации
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(self.device), labels.to(self.device)

                    outputs = model(images)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_acc = 100 * val_correct / val_total

            # Обновление шедулера
            scheduler.step()

            # Логирование
            logger.info(
                f"Эпоха {epoch+1}/{num_epochs}: "
                f"Train Loss: {train_loss/len(train_loader):.4f}, "
                f"Train Acc: {train_acc:.2f}%, "
                f"Val Loss: {val_loss/len(val_loader):.4f}, "
                f"Val Acc: {val_acc:.2f}%, "
                f"LR: {scheduler.get_last_lr()[0]:.6f}"
            )

            # Сохранение лучшей модели
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0

                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_accuracy': val_acc,
                    'train_accuracy': train_acc,
                    'classes': train_dataset.classes
                }, model_save_path)

                logger.info(f"Лучшая модель сохранена с точностью: {val_acc:.2f}%")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    logger.info(f"Ранняя остановка на эпохе {epoch+1}")
                    break

        logger.info(f"Обучение завершено. Лучшая точность: {best_val_acc:.2f}%")


def main() -> None:
    """Основная функция для демонстрации работы."""

    print("=" * 60)
    print("ПРОДВИНУТЫЙ КЛАССИФИКАТОР МАРОК АВТОМОБИЛЕЙ")
    print("=" * 60)

    # Инициализация классификатора
    config = ModelConfig(
        model_type=ModelType.EFFICIENTNET_B4,
        img_size=384,
        batch_size=16
    )

    # Проверка наличия обученных моделей
    model_paths = []
    if os.path.exists('best_model.pth'):
        model_paths.append('best_model.pth')

    use_ensemble = len(model_paths) > 1

    classifier = CarBrandClassifier(
        model_config=config,
        model_paths=model_paths if model_paths else None,
        use_ensemble=use_ensemble
    )

    # Пример использования
    while True:
        print("\n" + "=" * 40)
        print("МЕНЮ:")
        print("=" * 40)
        print("1. Распознать автомобиль по локальному изображению")
        print("2. Распознать автомобиль по URL изображения")
        print("3. Пакетное распознавание")
        print("4. Визуализировать предсказания")
        print("5. Обучить модель (требуется датасет)")
        print("6. Оценить модель (требуется тестовый набор)")
        print("7. Выйти")
        print("=" * 40)

        choice = input("\nВведите номер опции: ").strip()

        if choice == '1':
            image_path = input("Введите путь к изображению: ").strip()
            if os.path.exists(image_path):
                top_k = int(input("Количество лучших предсказаний (по умолчанию 3): ") or "3")
                confidence = float(input("Порог уверенности (0-100, по умолчанию 0): ") or "0")

                results = classifier.predict(image_path, top_k=top_k, confidence_threshold=confidence)

                if results:
                    print("\n" + "=" * 40)
                    print("РЕЗУЛЬТАТЫ РАСПОЗНАВАНИЯ:")
                    print("=" * 40)
                    for brand, prob in results:
                        print(f"  {brand.upper():<15} : {prob:>6.2f}%")

                    if results[0][1] > 80:
                        print(f"\n✓ Высокая уверенность: {results[0][0].upper()}")
                    elif results[0][1] > 60:
                        print(f"\n○ Средняя уверенность: {results[0][0].upper()}")
                    else:
                        print(f"\n⚠ Низкая уверенность, проверьте изображение")
                else:
                    print("Не удалось распознать изображение")
            else:
                print("Файл не найден")

        elif choice == '2':
            image_url = input("Введите URL изображения: ").strip()
            top_k = int(input("Количество лучших предсказаний (по умолчанию 3): ") or "3")

            results = classifier.predict_from_url(image_url, top_k=top_k)

            if results:
                print("\n" + "=" * 40)
                print("РЕЗУЛЬТАТЫ РАСПОЗНАВАНИЯ:")
                print("=" * 40)
                for brand, prob in results:
                    print(f"  {brand.upper():<15} : {prob:>6.2f}%")
            else:
                print("Не удалось распознать изображение")

        elif choice == '3':
            dir_path = input("Введите путь к папке с изображениями: ").strip()
            if os.path.exists(dir_path):
                image_paths = []
                for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                    image_paths.extend([os.path.join(dir_path, f) for f in os.listdir(dir_path)
                                      if f.lower().endswith(tuple(ext[1:]))])

                if image_paths:
                    print(f"\nНайдено {len(image_paths)} изображений")
                    batch_size = int(input("Размер батча (по умолчанию 16): ") or "16")

                    results = classifier.predict_batch(image_paths, batch_size=batch_size)

                    if results:
                        # Сохранение результатов
                        save_csv = input("Сохранить результаты в CSV? (y/n): ").strip().lower()
                        if save_csv == 'y':
                            classifier.save_predictions(results)

                        # Вывод статистики
                        print(f"\nОбработано {len(results)} изображений")

                        # Анализ распределения предсказаний
                        brand_counts = {}
                        for preds in results.values():
                            if preds:
                                brand = preds[0][0]
                                brand_counts[brand] = brand_counts.get(brand, 0) + 1

                        if brand_counts:
                            print("\nРаспределение марок:")
                            for brand, count in sorted(brand_counts.items(), key=lambda x: x[1], reverse=True):
                                percentage = (count / len(results)) * 100
                                print(f"  {brand.upper():<15}: {count:>3} ({percentage:.1f}%)")
                else:
                    print("Изображения не найдены")
            else:
                print("Папка не найдена")

        elif choice == '4':
            image_path = input("Введите путь к изображению: ").strip()
            if os.path.exists(image_path):
                save_viz = input("Сохранить визуализацию? (y/n): ").strip().lower()
                save_path = None
                if save_viz == 'y':
                    save_path = input("Путь для сохранения (по умолчанию prediction_vis.png): ").strip()
                    if not save_path:
                        save_path = 'prediction_vis.png'

                classifier.visualize_predictions(image_path, save_path)
            else:
                print("Файл не найден")

        elif choice == '5':
            print("\nОбучение модели требует подготовленного датасета.")
            print("Структура датасета:")
            print("  dataset/train/[brand_name]/images.jpg")
            print("  dataset/val/[brand_name]/images.jpg")

            dataset_path = input("Введите путь к датасету: ").strip()
            train_dir = os.path.join(dataset_path, 'train')
            val_dir = os.path.join(dataset_path, 'val')

            if os.path.exists(train_dir) and os.path.exists(val_dir):
                epochs = int(input("Количество эпох (по умолчанию 50): ") or "50")
                model_name = input("Имя модели для сохранения (по умолчанию best_model.pth): ") or "best_model.pth"

                trainer = AdvancedTrainer(config)
                trainer.train(train_dir, val_dir, num_epochs=epochs, model_save_path=model_name)
            else:
                print("Не найдены папки train и/или val")

        elif choice == '6':
            test_dir = input("Введите путь к тестовому набору данных: ").strip()
            if os.path.exists(test_dir):
                metrics = classifier.evaluate_model(test_dir)

                print(f"\nРезультаты оценки:")
                print(f"Точность: {metrics['accuracy']:.4f}")
                print(f"Precision: {metrics['precision']:.4f}")
                print(f"Recall: {metrics['recall']:.4f}")
                print(f"F1-Score: {metrics['f1_score']:.4f}")

                # Вывод отчета по классам
                report = metrics['classification_report']
                if 'weighted avg' in report:
                    print("\nСводные метрики:")
                    print(f"  Precision: {report['weighted avg']['precision']:.4f}")
                    print(f"  Recall: {report['weighted avg']['recall']:.4f}")
                    print(f"  F1-Score: {report['weighted avg']['f1-score']:.4f}")
            else:
                print("Тестовый набор не найден")

        elif choice == '7':
            print("Выход...")
            break

        else:
            print("Неверный выбор, попробуйте снова")


if __name__ == "__main__":
    # Создание необходимых директорий
    os.makedirs('models', exist_ok=True)
    os.makedirs('logs', exist_ok=True)
    os.makedirs('visualizations', exist_ok=True)

    main()